# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Ranking / Scoring

The goal is to help content editors decide which pages should be updated first. Instead of predicting a category, the system assigns a priority score to each content page using its SEO, engagement, and content features. Pages are then ranked from highest to lowest priority, allowing the content team to focus on the pages that are likely to benefit most from optimization.


In [11]:
# This notebook section is framed as a ranking/scoring problem.
# The output is a priority score for each page, then a ranked queue.

# Example framing for the lane


import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())
print(df.shape)
df.head()
print("Number of content items:", df["content_id"].nunique())
print("Number of clients:", df["client_id"].nunique())
performance_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "trend_direction"
]

df[performance_cols].head()




['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
(30000, 44)
Number of content items: 30000
Number of clients: 32


,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,trend_direction
0,3803,29,17,0.76,10.6,down
1,15320,7,9,0.05,20.3,down
2,12581,11,11,0.09,36.5,down
3,11751,58,78,0.49,6.2,stable
4,19140,24,145,0.13,44.0,down


In [12]:
ml_task_type = "Ranking / Scoring"

reason = (
    "Each row represents one content page with SEO and performance metrics. "
    "The goal is to prioritize which pages should be refreshed first, "
    "so the output should be a priority score and a ranked queue."
)

print("ML Task:", ml_task_type)
print(reason)

ML Task: Ranking / Scoring
Each row represents one content page with SEO and performance metrics. The goal is to prioritize which pages should be refreshed first, so the output should be a priority score and a ranked queue.


In [13]:
priority_pages = (
    df[
        [
            "content_id",
            "ctr",
            "avg_position",
            "trend_direction",
            "trend_pct"
        ]
    ]
    .sort_values(by="trend_pct")
    .head(10)
)

priority_pages

,content_id,ctr,avg_position,trend_direction,trend_pct
12149,content_d1d1eefe8db4,0.00,15.4,down,-100.0
16274,content_ded21d6ab93b,0.42,8.0,down,-100.0
1413,content_b48762a2c443,0.00,6.4,down,-100.0
16273,content_a9826bd63b56,0.00,3.0,down,-100.0
1410,content_1da78cd4c939,0.00,5.3,down,-100.0
23072,content_a56c7e0238af,0.00,16.0,down,-100.0
19412,content_2481ca68fb56,0.00,10.9,down,-100.0
27806,content_7585e0f05e8b,0.00,15.8,down,-100.0
28686,content_5d33ef77eb0f,0.00,11.6,down,-100.0
22019,content_8535cfede35b,0.00,26.7,down,-100.0


In [14]:
ml_task_type = "Ranking / Scoring"

reason = (
    "The dataset contains SEO and engagement metrics for each content page. "
    "Pages with declining trends and weaker performance can be prioritized "
    "for refresh. Therefore, the ML task is to assign each page a priority "
    "score and rank pages for review."
)

print(ml_task_type)
print(reason)

Ranking / Scoring
The dataset contains SEO and engagement metrics for each content page. Pages with declining trends and weaker performance can be prioritized for refresh. Therefore, the ML task is to assign each page a priority score and rank pages for review.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** A priority score representing how urgently a content page should be refreshed.

Since the starter dataset does not contain a direct priority score, a proxy target can be derived from observed historical performance, such as declining traffic and engagement over time. The score is based on observed outcomes rather than an arbitrary rule, allowing pages to be ranked from highest to lowest refresh priority.

In [16]:
target_cols = [
    "content_id",
    "trend_direction",
    "trend_pct",
    "clicks_90d",
    "sessions_90d"
]

df[target_cols].head()

,content_id,trend_direction,trend_pct,clicks_90d,sessions_90d
0,content_304f48230142,down,-41.4,29,17
1,content_a1fb4e703a9e,down,-57.7,7,9
2,content_9aa793d4d895,down,-60.9,11,11
3,content_331d6c4de07b,stable,-13.8,58,78
4,content_d99b7a2d90ca,down,-34.7,24,145


In [17]:
df["refresh_priority_proxy"] = (
    -df["trend_pct"] +
    (100 - df["ctr"]) +
    df["avg_position"]
)

df[["content_id", "refresh_priority_proxy"]].head()

,content_id,refresh_priority_proxy
0,content_304f48230142,151.24
1,content_a1fb4e703a9e,177.95
2,content_9aa793d4d895,197.31
3,content_331d6c4de07b,119.51
4,content_d99b7a2d90ca,178.57


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric:** Precision@K

Precision@K measures how many of the top-ranked pages are truly high-priority pages for refresh. A good model places the pages that most need attention near the top of the ranked list, allowing the content team to spend their time on the most valuable updates first.

In [18]:
# Pages with the largest performance decline
df[
    [
        "content_id",
        "trend_direction",
        "trend_pct",
        "ctr",
        "avg_position"
    ]
].sort_values("trend_pct").head(10)

,content_id,trend_direction,trend_pct,ctr,avg_position
12149,content_d1d1eefe8db4,down,-100.0,0.00,15.4
16274,content_ded21d6ab93b,down,-100.0,0.42,8.0
1413,content_b48762a2c443,down,-100.0,0.00,6.4
16273,content_a9826bd63b56,down,-100.0,0.00,3.0
1410,content_1da78cd4c939,down,-100.0,0.00,5.3
23072,content_a56c7e0238af,down,-100.0,0.00,16.0
19412,content_2481ca68fb56,down,-100.0,0.00,10.9
27806,content_7585e0f05e8b,down,-100.0,0.00,15.8
28686,content_5d33ef77eb0f,down,-100.0,0.00,11.6
22019,content_8535cfede35b,down,-100.0,0.00,26.7


In [19]:
df["trend_direction"].value_counts()

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

In [20]:
success_metric = "Precision@K"

print(f"Chosen success metric: {success_metric}")
print("Precision@K will be calculated after the model produces a ranked list of pages.")

Chosen success metric: Precision@K
Precision@K will be calculated after the model produces a ranked list of pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis

Each row represents one content item (one page/article) for a client. The row contains the page's SEO characteristics, content attributes, and historical performance metrics over the previous 90 days. This is the unit that will receive a priority score and be ranked for content refresh.

In [21]:
df[[
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "word_count",
    "ctr",
    "avg_position",
    "trend_direction"
]].head()

,content_id,client_id,content_type,main_intent,word_count,ctr,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,0.76,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,0.05,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,0.09,36.5,down
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,0.49,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,0.13,44.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule cannot capture the complex relationships between SEO performance, content quality, user engagement, content freshness, and search competition. A page may have a low CTR because of poor ranking, low search volume, or outdated content, while another page with a similar CTR may not need immediate attention.

A ranking model can learn from many interacting signals at the same time and produce a better priority score than a single manually defined rule. This helps the content team focus on the pages that are most likely to benefit from a refresh.

In [22]:
signals = [
    "search_volume",
    "competition",
    "word_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days",
    "trend_direction"
]

df[signals].head()

,search_volume,competition,word_count,ctr,avg_position,engagement_rate,content_age_days,trend_direction
0,10.0,0.67,3221.0,0.76,10.6,5.88,187,down
1,90.0,0.01,2481.0,0.05,20.3,0.00,445,down
2,0.0,0.00,3515.0,0.09,36.5,0.00,141,down
3,10.0,0.00,NaN,0.49,6.2,1.28,463,stable
4,0.0,0.00,2803.0,0.13,44.0,0.00,263,down


In [23]:
print(f"Number of signals considered: {len(signals)}")
print("Signals:", signals)

Number of signals considered: 8
Signals: ['search_volume', 'competition', 'word_count', 'ctr', 'avg_position', 'engagement_rate', 'content_age_days', 'trend_direction']


In [24]:
print(
    "Refresh priority depends on multiple SEO, engagement, and content signals, "
    "so a single if-statement is unlikely to capture the full pattern."
)

Refresh priority depends on multiple SEO, engagement, and content signals, so a single if-statement is unlikely to capture the full pattern.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.